# Extract ACCESS-OM2 Emulator Data

This notebook builds the combined ACCESS-OM2 dataset used by the emulator workflow.
It extracts tracer-cell area, total surface heat flux, surface wind stress (tau_x, tau_y), and vertically integrated ocean heat content from the `1deg_jra55_iaf_omip2_cycle6` run.

`tau_x`/`tau_y` are diagnosed on the model's u-cell (velocity) grid (`xu_ocean`/`yu_ocean`), one half-gridcell offset from the t-cell (tracer) grid used by everything else in this dataset. They are bilinearly interpolated onto the t-cell grid (`xt_ocean`/`yt_ocean`) before saving, so every variable in the emulator dataset shares one common grid.

The source archive spans 1958-01 to 2018-12. The defaults below therefore build the full available period into a single emulator dataset file.

In [ ]:
from pathlib import Path
import warnings

from dask.distributed import Client
from tqdm.auto import tqdm
import xarray as xr

warnings.filterwarnings("ignore")

In [ ]:
EXPERIMENT = "1deg_jra55_iaf_omip2_cycle6"
START_DATE = "1958-01-01"
END_DATE = "2018-12-31"
OUTPUT_PATH = "/g/data/dx2/rmh561/OM2-emulator/data/1deg_ocean_heat_emulator_data_1958_2018.nc"

REQUIRED_VARIABLES = ["net_sfc_heating", "frazil_3d_int_z", "temp", "rho_dzt", "rho", "tau_x", "tau_y"]
CP_SEAWATER = 3992.10322329649
RHO0 = 1035.0
TIME_CHUNK = 12
DEPTH_CHUNK = 10

In [ ]:
client = Client()
client

archive_root = Path(f"/g/data/ik11/outputs/access-om2/{EXPERIMENT}")
ocean_month_files = sorted(
    str(path / "ocean" / "ocean_month.nc")
    for path in archive_root.iterdir()
    if path.name.startswith("output") and (path / "ocean" / "ocean_month.nc").exists()
 )

source_ds = xr.open_mfdataset(
    ocean_month_files,
    combine="by_coords",
    use_cftime=True,
    chunks={"time": TIME_CHUNK, "st_ocean": DEPTH_CHUNK},
)[[*REQUIRED_VARIABLES]]

source_ds = source_ds.sel(time=slice(START_DATE, END_DATE)).sortby("time")
area_t = xr.open_dataset(
    archive_root / "output305" / "ocean" / "ocean_grid.nc",
    decode_times=False,
)["area_t"].load()

available_years = sorted({int(timestamp.year) for timestamp in source_ds.time.values})
yearly_output_dir = Path(OUTPUT_PATH).with_suffix("").parent / f"{Path(OUTPUT_PATH).stem}_by_year"
yearly_output_dir.mkdir(parents=True, exist_ok=True)

print(
    f"Loaded source period: {source_ds.time.values[0]} to {source_ds.time.values[-1]} "
    f"({source_ds.sizes['time']} monthly steps across {len(available_years)} years)"
)

In [ ]:
def regrid_u_to_t(data_array, target_ds):
    """Bilinearly interpolate a u-cell (xu_ocean/yu_ocean) field onto the
    t-cell (xt_ocean/yt_ocean) grid of `target_ds`.

    ACCESS-OM2's wind stress diagnostics (tau_x, tau_y) are staggered by half
    a gridcell from the tracer grid used by every other variable in this
    dataset. Assigning the t-grid's own coordinate arrays as the interpolation
    targets both regrids the field and relabels its dimensions to
    (..., yt_ocean, xt_ocean) in one step.
    """
    return data_array.interp(
        xu_ocean=target_ds["xt_ocean"], yu_ocean=target_ds["yt_ocean"]
    )


def build_emulator_year(year):
    year_ds = source_ds.sel(time=str(year))

    total_surface_heat_flx = year_ds["net_sfc_heating"] + year_ds["frazil_3d_int_z"]
    total_surface_heat_flx = total_surface_heat_flx.assign_attrs(
        long_name=f"Total surface heat flux including frazil ice from {EXPERIMENT}",
        units=year_ds["net_sfc_heating"].attrs.get("units", "W m-2"),
        description="net_sfc_heating + frazil_3d_int_z",
    )
    total_surface_heat_flx.name = "total_surface_heat_flx"

    layer_thickness = year_ds["rho_dzt"] / year_ds["rho"]
    ocean_heat_content_2d = (year_ds["temp"] * layer_thickness).sum("st_ocean") * CP_SEAWATER * RHO0
    ocean_heat_content_2d = ocean_heat_content_2d.assign_attrs(
        long_name=f"Vertically integrated ocean heat content from {EXPERIMENT}",
        units="J/m2",
        description="(temp * (rho_dzt / rho)).sum(st_ocean) * cp * rho0",
    )
    ocean_heat_content_2d.name = "ocean_heat_content_2d"

    # tau_x/tau_y are diagnosed on the u-cell grid; regrid onto the t-cell grid
    # so every saved variable shares the same (yt_ocean, xt_ocean) grid.
    tau_x = regrid_u_to_t(year_ds["tau_x"], year_ds).assign_attrs(year_ds["tau_x"].attrs)
    tau_x.name = "tau_x"
    tau_y = regrid_u_to_t(year_ds["tau_y"], year_ds).assign_attrs(year_ds["tau_y"].attrs)
    tau_y.name = "tau_y"

    return xr.Dataset(
        {
            "area_t": area_t,
            "total_surface_heat_flx": total_surface_heat_flx,
            "ocean_heat_content_2d": ocean_heat_content_2d,
            "tau_x": tau_x,
            "tau_y": tau_y,
        }
    )


yearly_files = []
for year in tqdm(available_years, desc="Writing yearly files", unit="year"):
    year_path = yearly_output_dir / f"{Path(OUTPUT_PATH).stem}_{year}.nc"
    build_emulator_year(year).to_netcdf(year_path)
    yearly_files.append(str(year_path))

emulator_ds = xr.open_mfdataset(
    yearly_files,
    combine="by_coords",
    use_cftime=True,
    chunks={"time": TIME_CHUNK},
)

emulator_ds

In [ ]:
emulator_ds.to_netcdf(OUTPUT_PATH)

print(f"Saved emulator dataset to: {OUTPUT_PATH}")
print(
    f"Saved period: {emulator_ds.time.values[0]} to {emulator_ds.time.values[-1]} "
    f"({emulator_ds.sizes['time']} monthly steps)"
)
print(f"Intermediate yearly files are in: {yearly_output_dir}")